In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from moabb.paradigms import MotorImagery
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation

datasets = [
    #AlexMI(),
    BNCI2014_001(),
    #PhysionetMI(),
    #Schirrmeister2017(),
    #Weibo2014(),
    #Zhou2016()
]

n_classes=3
sfreq=250

In [3]:
import copy
from sklearn.base import clone
import dask
import os
import tensorly as tl
from classification_mi import stf_transform
from sklearn.preprocessing import FunctionTransformer

cache_config = dict(
    use=True,
    save_raw=False,
    save_epochs=False,
    save_array=True,
    overwrite_raw=False,
    overwrite_epochs=False,
    overwrite_array=False,
)

def eval_moabb_within_session(dataset, subject, pipe):
    subj_dataset = copy.deepcopy(dataset)

    events = list(subj_dataset.event_id.keys())[:n_classes]
    paradigm = MotorImagery(events=events, n_classes=n_classes, resample=sfreq)

    subj_dataset = copy.deepcopy(dataset)
    n_subjects = len(dataset.subject_list)
    subj_dataset.subject_list = [subject]
    
    evaluation = WithinSessionEvaluation(
        paradigm=paradigm,
        datasets=subj_dataset,
        overwrite=True,
        random_state=42,
        n_jobs=-1,
        suffix=f'bttda_dask_dataset-{dataset.code}_subject-{subject}_pipe-{pipe}',
        cache_config=cache_config,
    )
    print(f'dataset={dataset.code}, subject={subject}/{n_subjects}, pipe={pipe}')
    return evaluation.process(
        {pipe:clone(pipelines[pipe])},
        postprocess_pipeline= FunctionTransformer(stf_transform)
    )





In [4]:
from classification_mi import get_pipelines_mi
pipelines = get_pipelines_mi()
pipelines


{'HODA': Pipeline(steps=[('tensorly',
                  FunctionTransformer(func=<function NumpyBackend.tensor at 0x14f5136b36a0>)),
                 ('zscore1', ZScore()),
                 ('bttda',
                  BTTDACV(clf=Pipeline(steps=[('functiontransformer',
                                               FunctionTransformer(func=<function NumpyBackend.to_numpy at 0x14f5136b32e0>)),
                                              ('pca', PCA(whiten=True)),
                                              ('selectfcutoff', SelectFCutoff()),
                                              ('lineardiscriminantanalysis',
                                               Line...
                          max_n_blocks=1, n_jobs=-1,
                          thetas=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8,
                                  0.9, 1.0])),
                 ('clf',
                  Pipeline(steps=[('functiontransformer',
                                   FunctionTransformer(

In [5]:
import joblib
from joblib import Parallel, delayed
import distributed
from IPython import display
import pandas as pd
from hpc import create_cluster, create_client, TIMEOUT

with create_cluster(cluster='cpu') as cluster, create_client(cluster) as client:
    results = []
    for dataset in datasets:
        print(f'Benchmarking on dataset {dataset.code}...')
        job_args = []
        for subject in dataset.subject_list:
            for pipe in pipelines.keys():
                job_args.append((dataset, subject,pipe))    
        with joblib.parallel_backend('dask', wait_for_workers_timeout=TIMEOUT): 
            results += Parallel(n_jobs=-1, verbose=True)(delayed(eval_moabb_within_session)(*args) for args in job_args)
results = pd.concat(results, ignore_index=True)

Benchmarking on dataset BNCI2014-001...


[Parallel(n_jobs=-1)]: Using backend DaskDistributedBackend with 18 concurrent workers.
2025-08-21 13:44:15,690 - distributed.scheduler - ERROR - Removing worker 'tcp://10.118.228.200:36497' caused the cluster to lose scattered data, which can't be recovered: {'ndarray-510e26323d5e4d9097b466d5d1d07dca', 'ndarray-ee6c828f8b27475b9d0c665a97ef8a9c', 'ndarray-d197def92bbe4c1791a201d021f17717', 'ndarray-5aa233285a364898b2615d738c54bf39', 'ndarray-cc0734a70d6042fa8d03b57ed5714f35', 'ndarray-7e4dbf42f3ba4b1ab64c5bfe85b0cc21', 'ndarray-8803fb5589cf41568c2dd81e593788a3', 'ndarray-03a6ce22fc954c6caf820ec94c3b6c01', 'ndarray-86284073bab14ad2b46bb9404387acbe', 'ndarray-c9a5afd78f9c4f00afdcc7a4e089b646', 'ndarray-1c8ce14542254e509cbd598d98c20a56', 'ndarray-dc22b1a8e25649a1bd3b0362e19c3748', 'ndarray-113eed20348f443f9fa3fe0b6013ea73', 'ndarray-5cfdc89717044e4fa1f62aee64586ba3', 'ndarray-213c11d4aae34f3c9da18e5b265e5a8d', 'ndarray-65f47d6c3ff34d359f331392cc6689d5', 'ndarray-b8002a9c40a9464cab2f6705b1

RuntimeError: Did not find any scans.tsv associated with sub-8_ses-0train_task-imagery_run-0_desc-28d79f636e4a15089119bc5e2f4cc758.

The search_str was "/scratch/leuven/352/vsc35289/mne_data/MNE-BIDS-bnci2014-001/sub-8/**/sub-8_ses-0train*scans.tsv"

In [ ]:
results.to_csv('results/moabb_mi.csv')
results

In [ ]:
results = pd.read_csv('results/moabb_mi.csv')

In [ ]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate(['mean', 'std'])

In [ ]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate('mean').reset_index().groupby('pipeline')['score'].aggregate('mean')

In [ ]:
df_diff = results.pivot(index=['subject', 'session', 'channels', 'n_sessions', 'samples', 'dataset'], columns='pipeline', values='score')
df_diff = df_diff.reset_index()
df_diff['score_diff'] = df_diff['BTTDA'] - df_diff['HODA']
df_diff

In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'iframe'

def compare_score_plot(df, pipe1, pipe2):
    fig = px.scatter(df, x=pipe1, y=pipe2, color='dataset', facet_col='dataset', facet_col_wrap=5)
    fig.update_yaxes(scaleanchor="x")
    fig.update_xaxes(range=[0, 1])
    fig.update_yaxes(range=[0, 1])
    fig.add_shape(
        type="line",
        x0=0, y0=0.0, x1=1, y1=1,
        line=dict(color="gray", dash='dash'),
        layer="below" ,
        row='all', col='all', exclude_empty_subplots=True
    )
    

    return fig

fig = compare_score_plot(df_diff, 'HODA', 'BTTDA')
fig.update_layout(
    autosize=False,
    width=1800,
    height=1800,
)
fig.update_layout(showlegend=False)
fig